In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path
from tqdm import tqdm
from collections import deque
import warnings
warnings.filterwarnings('ignore')


class SmartAdaptiveExtractor:
    def __init__(self):
        self.detection_history        = deque(maxlen=10)
        self.hand_size_history        = deque(maxlen=10)
        self.frames_since_detection   = 0
        self.stats = {
            'normal'  : 0,
            'adaptive': 0,
            'failed'  : 0,
            'total'   : 0,
        }

    def _hand_size(self, results, h, w):
        sizes = []
        for lm_set in [results.left_hand_landmarks,
                        results.right_hand_landmarks]:
            if lm_set:
                xs = [lm.x * w for lm in lm_set.landmark]
                ys = [lm.y * h for lm in lm_set.landmark]
                sizes.append(max(max(xs)-min(xs), max(ys)-min(ys)))
        return np.mean(sizes) if sizes else 0

    def _has_hands(self, results):
        return bool(results.left_hand_landmarks
                    or results.right_hand_landmarks)

    def _scale_back(self, results, scale):
        for lm_set in [results.pose_landmarks,
                        results.left_hand_landmarks,
                        results.right_hand_landmarks]:
            if lm_set:
                for lm in lm_set.landmark:
                    lm.x /= scale
                    lm.y /= scale
        return results

    def extract(self, frame, holistic):
        h, w = frame.shape[:2]
        self.stats['total'] += 1

        # Try normal first
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        if self._has_hands(results):
            sz = self._hand_size(results, h, w)
            self.hand_size_history.append(sz)
            self.frames_since_detection = 0
            self.stats['normal'] += 1
            return results

        # Decide scale based on history
        self.frames_since_detection += 1
        avg_sz     = np.mean(self.hand_size_history) \
                     if self.hand_size_history else 0
        scales     = [1.3, 1.6, 2.0] if avg_sz < 80 else [1.3]

        for scale in scales:
            if scale * min(h, w) > 2000:
                continue
            up  = cv2.resize(frame, (int(w*scale), int(h*scale)),
                             interpolation=cv2.INTER_LINEAR)
            res = holistic.process(cv2.cvtColor(up, cv2.COLOR_BGR2RGB))
            if self._has_hands(res):
                self._scale_back(res, scale)
                self.stats['adaptive'] += 1
                return res

        self.stats['failed'] += 1
        return results

    def get_stats(self):
        t = max(self.stats['total'], 1)
        return {
            'normal_detection_rate'  : self.stats['normal']   / t,
            'adaptive_detection_rate': self.stats['adaptive'] / t,
            'failed_detection_rate'  : self.stats['failed']   / t,
        }


# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_pose(pose_lm):
    if pose_lm is None:
        return np.zeros(132, dtype=np.float32)
    arr = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                     for lm in pose_lm.landmark], dtype=np.float32)
    # Normalize relative to hip midpoint
    hip = (arr[23, :3] + arr[24, :3]) / 2
    arr[:, :3] -= hip
    return arr.flatten()

def extract_hand(hand_lm):
    if hand_lm is None:
        return np.zeros(63, dtype=np.float32)
    pts = np.array([[lm.x, lm.y, lm.z]
                     for lm in hand_lm.landmark], dtype=np.float32)
    return (pts - pts[0]).flatten()

def get_windows(total, size, step):
    wins = []
    for s in range(0, total - size + 1, step):
        wins.append(list(range(s, s + size)))
    # Always include last window
    last = total - size
    if not wins or wins[-1][0] != last:
        wins.append(list(range(last, last + size)))
    return wins

def pad_seq(arr, target):
    if arr.shape[0] >= target:
        return arr
    pad = np.tile(arr[-1:], (target - arr.shape[0], 1))
    return np.concatenate([arr, pad])


# ============================================================
# MAIN
# ============================================================
def extract_landmarks_smart():

    BASE_DIR     = Path(r"D:\uni\Intern-1-Project\ksl")
    DATA_DIR     = BASE_DIR / 'dataset' / 'education words' / '24 sign'
    LANDMARK_DIR = BASE_DIR / 'landmarks_30frames_smart_old'

    SELECTED_CLASSES = [
        'កុំព្យូទ័រ','កៅអី','ក្ដារខៀន','ខ្មៅដៃ','ជ័រលុប','ដីស','តុ','ទឹកលុប','នាយករង','នាយិកា','បន្ទាត់','សៀវភៅ','ប៊ិកខៀវ','ហ្វឺតខ្មៅ',
        'ហ្វឺតក្រហម','ប៊ិក','ប៊ិកក្រហម','កាតាប','កាតាបស្ពាយក្រោយ','ហ្វឺតខៀវ'
    ]

    SEQ_LEN      = 30
    WINDOW_STEP  = 15
    USE_POSE     = True
    USE_HANDS    = True   # both hands
    USE_VELOCITY = True

    # ── Feature sizes ──────────────────────────────────────────
    POSE_FEAT  = 132 if USE_POSE  else 0
    HANDS_FEAT = 126 if USE_HANDS else 0  # 63 × 2 (left + right)
    BASE_FEAT  = POSE_FEAT + HANDS_FEAT
    TOTAL_FEAT = BASE_FEAT * 2 if USE_VELOCITY else BASE_FEAT

    print("="*60)
    print("SMART ADAPTIVE EXTRACTION")
    print("="*60)
    print(f"Pose features  : {POSE_FEAT}")
    print(f"Hands features : {HANDS_FEAT}  (left 63 + right 63)")
    print(f"Base per frame : {BASE_FEAT}")
    print(f"Total w/vel    : {TOTAL_FEAT}")
    print(f"Output shape   : ({SEQ_LEN}, {TOTAL_FEAT})")
    print("="*60)

    LANDMARK_DIR.mkdir(parents=True, exist_ok=True)

    all_classes = sorted([d.name for d in DATA_DIR.iterdir()
                           if d.is_dir()])
    targets     = [c for c in SELECTED_CLASSES if c in all_classes]

    if not targets:
        print(" No valid classes found!")
        print("Available:", all_classes[:10])
        return

    mp_holistic = mp.solutions.holistic

    for class_name in targets:
        class_path = DATA_DIR / class_name
        save_path  = LANDMARK_DIR / class_name
        save_path.mkdir(parents=True, exist_ok=True)

        # Skip if already done
        done = len(list(save_path.glob('*.npy')))
        if done > 0:
            print(f"Skipping {class_name} ({done} files exist)")
            continue

        videos  = sorted(list(class_path.glob('*.mp4')) +
                          list(class_path.glob('*.avi')))
        print(f"\n {class_name}: {len(videos)} videos")

        extractor      = SmartAdaptiveExtractor()
        sample_counter = 0
        hand_detected  = 0
        videos_ok      = 0

        with mp_holistic.Holistic(
            static_image_mode        = False,
            model_complexity         = 1,
            min_detection_confidence = 0.3,
            min_tracking_confidence  = 0.3
        ) as holistic:

            for v_idx, video_file in enumerate(
                    tqdm(videos, desc=class_name)):

                cap    = cv2.VideoCapture(str(video_file))
                frames = []
                while True:
                    ret, frame = cap.read()
                    if not ret: break
                    frames.append(frame)
                cap.release()

                if len(frames) < SEQ_LEN // 2:
                    continue

                # ── Extract landmarks per frame ───────────────
                all_feats = []
                valid     = 0

                for frame in frames:
                    res = extractor.extract(frame, holistic)

                    pose_f  = extract_pose(res.pose_landmarks)
                    left_f  = extract_hand(res.left_hand_landmarks)
                    right_f = extract_hand(res.right_hand_landmarks)

                    combined = np.concatenate([pose_f, left_f, right_f])
                    # shape: (258,) = 132 + 63 + 63

                    has_hand = (res.left_hand_landmarks is not None
                                or res.right_hand_landmarks is not None)
                    if has_hand:
                        valid      += 1
                        hand_detected += 1

                    all_feats.append(combined)

                if valid == 0:
                    continue

                videos_ok += 1

                # ── Create overlapping windows ────────────────
                total   = len(all_feats)
                windows = get_windows(total, SEQ_LEN, WINDOW_STEP)

                for win in windows:
                    feats = [all_feats[i] for i in win
                             if i < len(all_feats)]
                    if len(feats) < SEQ_LEN // 2:
                        continue

                    pos = np.array(feats, dtype=np.float32)

                    # Pad if needed
                    if pos.shape[0] < SEQ_LEN:
                        pos = pad_seq(pos, SEQ_LEN)

                    # Add velocity
                    if USE_VELOCITY:
                        vel     = np.zeros_like(pos)
                        vel[1:] = pos[1:] - pos[:-1]
                        out     = np.concatenate([pos, vel], axis=1)
                    else:
                        out = pos

                    # ── Shape check ───────────────────────────
                    if out.shape != (SEQ_LEN, TOTAL_FEAT):
                        print(f"   Shape mismatch: {out.shape} "
                              f"expected ({SEQ_LEN},{TOTAL_FEAT})")
                        continue

                    # ── Quality gate ──────────────────────────
                    zero_ratio = np.mean(
                        np.all(out[:, POSE_FEAT:POSE_FEAT+63] == 0,
                               axis=1)
                    )
                    if zero_ratio > 0.6:
                        continue  # too many missing hand frames

                    np.save(save_path/f'{sample_counter:06d}.npy', out)
                    sample_counter += 1

                if (v_idx+1) % 10 == 0:
                    print(f"  Video {v_idx+1}/{len(videos)} | "
                          f"Samples so far: {sample_counter}")

        # Summary
        stats = extractor.get_stats()
        print(f"\n {class_name}:")
        print(f"   Videos OK    : {videos_ok}/{len(videos)}")
        print(f"   Samples saved: {sample_counter}")
        print(f"   Hand detected: {hand_detected} frames")
        print(f"   Normal detect: {stats['normal_detection_rate']:.1%}")
        print(f"   Adaptive det : {stats['adaptive_detection_rate']:.1%}")
        print(f"   Failed       : {stats['failed_detection_rate']:.1%}")

    print("\n" + "="*60)
    print("DONE!")
    print(f"Output: {LANDMARK_DIR}")
    print(f"Shape : ({SEQ_LEN}, {TOTAL_FEAT})")
    print("="*60)


if __name__ == '__main__':
    extract_landmarks_smart()